# N05 — Figure 3: engineered-noise ensemble experiment

This notebook regenerates the manuscript's engineered-noise result directly from the 20 raw oscilloscope traces (10 fast + 10 slow).

The notebook is designed to make three distinctions explicit:

1. energy is reconstructed **trajectory by trajectory** before averaging;
2. the colored bands are run-to-run sample standard deviations, not confidence intervals;
3. the gray interval is a trajectory-level bootstrap interval for the crossing time.

The experiment uses externally injected engineered AWGN. No effective temperature is assigned.


In [ ]:
from pathlib import Path
import json, csv, re, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.linalg import expm

# Locate the reproducibility-bundle root whether the notebook is launched
# from the bundle root or from notebooks/.
def locate_root(start=Path.cwd()):
    candidates = [start, start.parent, start.parent.parent]
    for p in candidates:
        if (p / "experimental_data" / "raw").exists():
            return p.resolve()
    raise FileNotFoundError("Could not locate bundle root containing experimental_data/raw/." )

ROOT = locate_root()
RAW = ROOT / "experimental_data" / "raw"
CERT = ROOT / "experimental_data" / "certified"
OUT_DATA = ROOT / "notebook_outputs" / "data"
OUT_FIG = ROOT / "notebook_outputs" / "figures"
OUT_DATA.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

# ---------------- Plot-style switch ----------------
# Safe default: executes everywhere with Matplotlib only.
# Local options:
#   PLOT_STYLE = "science-no-latex"  -> requires `pip install scienceplots`
#   PLOT_STYLE = "science-latex"     -> additionally requires a local LaTeX install
PLOT_STYLE = "matplotlib"

def apply_plot_style(style=PLOT_STYLE):
    plt.style.use("default")
    if style.startswith("science"):
        try:
            import scienceplots  # noqa: F401
            plt.style.use(["science", "no-latex"] if style == "science-no-latex" else ["science"])
        except ImportError:
            warnings.warn("scienceplots is not installed; falling back to Matplotlib.")
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["STIXGeneral", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 9.0,
        "axes.labelsize": 9.0,
        "axes.titlesize": 9.5,
        "legend.fontsize": 7.5,
        "xtick.labelsize": 8.0,
        "ytick.labelsize": 8.0,
        "axes.linewidth": 0.7,
        "lines.linewidth": 1.3,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.04,
    })

apply_plot_style()

# Measured/component values used in the manuscript.
L_H = 44.4e-3
C_F = 454e-6
R_EXT_OHM = 22.2
R_COMP_OHM = 27.9
DETERMINISTIC_RELEASE_S = 7.2e-3

print(f"Bundle root: {ROOT}")
print(f"Plot style: {PLOT_STYLE}")


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_scope_csv(path):
    """Read the two Tektronix channel blocks stored side-by-side in one CSV."""
    path = Path(path)
    rows = list(csv.reader(path.open(newline="", encoding="utf-8", errors="ignore")))
    channels = {}
    for block_start in (0, 6):
        metadata = {}
        for row in rows[:20]:
            if len(row) > block_start + 1 and row[block_start].strip():
                metadata[row[block_start].strip()] = row[block_start + 1].strip()
        n = int(round(float(metadata["Record Length"])))
        dt = float(metadata["Sample Interval"])
        trigger = float(metadata["Trigger Point"])
        values = []
        for row in rows:
            if len(row) > block_start + 4:
                try:
                    values.append(float(row[block_start + 4]))
                except ValueError:
                    pass
        values = np.asarray(values, dtype=float)
        assert len(values) == n, (path.name, block_start, len(values), n)
        time_s = (np.arange(n, dtype=float) - trigger) * dt
        source = metadata.get("Source", f"CH{len(channels)+1}")
        channels[source] = {
            "time_s": time_s,
            "values": values,
            "metadata": metadata,
            "dt_s": dt,
        }
    assert set(channels) == {"CH1", "CH2"}
    assert np.allclose(channels["CH1"]["time_s"], channels["CH2"]["time_s"])
    return channels


def reconstruct_scope(path, release_shift_s=None):
    channels = read_scope_csv(path)
    t_trigger = channels["CH1"]["time_s"]
    vc = channels["CH1"]["values"]
    vr = channels["CH2"]["values"]
    current = -vr / R_EXT_OHM
    charge = C_F * vc
    energy = 0.5 * C_F * vc**2 + 0.5 * L_H * current**2
    frame = pd.DataFrame({
        "time_after_trigger_s": t_trigger,
        "Vc_V": vc,
        "Vr_V": vr,
        "q_C": charge,
        "I_A": current,
        "E_J": energy,
    })
    if release_shift_s is not None:
        frame.insert(1, "time_after_release_s", t_trigger - release_shift_s)
    return frame, channels


def first_crossing(t1, e1, t2, e2, start_s=0.0):
    """First sign change of e1-e2, with linear interpolation."""
    t1 = np.asarray(t1); e1 = np.asarray(e1); t2 = np.asarray(t2); e2 = np.asarray(e2)
    mask = (t1 >= start_s) & (t1 <= t2[-1])
    tx = t1[mask]
    d = e1[mask] - np.interp(tx, t2, e2)
    idx = np.where(np.diff(np.signbit(d)))[0]
    if len(idx) == 0:
        return np.nan, np.nan
    i = int(idx[0])
    tc = tx[i] - d[i] * (tx[i+1] - tx[i]) / (d[i+1] - d[i])
    ec = float(np.interp(tc, tx, e1[mask]))
    return float(tc), ec


## 1. Load and reconstruct every run


In [ ]:
NOISE=RAW/"engineered_noise"
files=sorted(NOISE.glob("*.csv"),key=lambda p:(0 if "rapido" in p.stem else 1,int(re.search(r"\d+",p.stem).group())))
paths={"fast":[p for p in files if "rapido" in p.stem],"slow":[p for p in files if "lento" in p.stem]}
assert len(paths["fast"])==len(paths["slow"])==10

frames={"fast":[],"slow":[]}; t=None
for mode in ["fast","slow"]:
    for p in paths[mode]:
        d,_=reconstruct_scope(p)
        if t is None: t=d["time_after_trigger_s"].to_numpy()
        assert np.allclose(t,d["time_after_trigger_s"])
        frames[mode].append(d)
print("raw runs loaded:", {k:len(v) for k,v in frames.items()})


## 2. Mean energy, run-to-run spread, and moment decomposition

For each time sample,
\[
\overline{E}_N=\frac12\mu_N^T G\mu_N+\frac12\operatorname{Tr}(G\Sigma_N)
\equiv E_\mu+E_\Sigma.
\]
The finite-ensemble covariance uses \(1/N\), which makes this identity exact for the same finite sample. The run-to-run standard deviation uses \(1/(N-1)\).


In [ ]:
def stats(fr):
    vc=np.vstack([d["Vc_V"] for d in fr]); cur=np.vstack([d["I_A"] for d in fr]); eng=np.vstack([d["E_J"] for d in fr])
    muv=vc.mean(0); mui=cur.mean(0)
    Emu=.5*C_F*muv**2+.5*L_H*mui**2
    Esig=.5*C_F*vc.var(0,ddof=0)+.5*L_H*cur.var(0,ddof=0)
    out={"eng":eng,"mean":eng.mean(0),"sd":eng.std(0,ddof=1),"Emu":Emu,"Esig":Esig}
    out["closure"]=float(np.max(np.abs(out["mean"]-Emu-Esig)))
    return out
S={k:stats(v) for k,v in frames.items()}
print("moment closure [J]",{k:v["closure"] for k,v in S.items()})
assert max(v["closure"] for v in S.values())<2e-19


## 3. Crossing and trajectory bootstrap


In [ ]:
tx,Ex=first_crossing(t,S["fast"]["mean"],t,S["slow"]["mean"],start_s=0)
SEED=20260908; B=50_000
rng=np.random.default_rng(SEED)
draws=rng.integers(0,10,size=(B,20))
m=(t>=0)&(t<=.008); tc=t[m]
Ef=S["fast"]["eng"][:,m]; Es=S["slow"]["eng"][:,m]
boots=np.empty(B)
for lo in range(0,B,1000):
    hi=min(lo+1000,B); d=draws[lo:hi]
    diff=Ef[d[:,:10]].mean(1)-Es[d[:,10:]].mean(1)
    changes=np.diff(np.signbit(diff),axis=1)
    assert changes.any(1).all()
    j=changes.argmax(1); rr=np.arange(hi-lo)
    d0=diff[rr,j]; d1=diff[rr,j+1]
    boots[lo:hi]=tc[j]-d0*(tc[j+1]-tc[j])/(d1-d0)
ci95=np.percentile(1e3*boots,[2.5,97.5])
print(f"crossing = {1e3*tx:.9f} ms")
print("95% bootstrap interval [ms] =",ci95)


## 4. Covariance fraction and late-time window

The plotted fraction is \(E_\Sigma/\overline{E}\). For readability, only this **display quantity** is time-binned; the underlying energies, crossing, and ensemble statistics are never smoothed.


In [ ]:
frac={mode:S[mode]["Esig"]/S[mode]["mean"] for mode in ["fast","slow"]}
late=(t>=.040)&(t<=.100)
late_avg={mode:float(S[mode]["Esig"][late].mean()/S[mode]["mean"][late].mean()) for mode in ["fast","slow"]}
print("40–100 ms covariance fractions:",late_avg)

edges=np.r_[np.arange(0,10.0001,.5),np.arange(11,40.0001,1.0),np.arange(42,100.0001,2.0)]
edges=np.unique(edges); tm=1e3*t

def bin_mean(x,y,edges):
    xc=[];yc=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        mm=(x>=lo)&(x<hi)
        if mm.any():
            xc.append(.5*(lo+hi));yc.append(np.nanmean(y[mm]))
    return np.array(xc),np.array(yc)

binned={mode:bin_mean(tm,frac[mode],edges) for mode in ["fast","slow"]}


## 5. Regenerate Figure 3

Panel (a) is the experimentally observed ensemble ordering inversion. Panel (b) shows that late-time mean energy is dominated by ensemble covariance rather than by the energy of the ensemble mean.


In [ ]:
cycle=mpl.rcParams["axes.prop_cycle"].by_key()["color"]
SLOW,FAST=cycle[0],cycle[1]
fig=plt.figure(figsize=(6.18,3.42))
gs=fig.add_gridspec(1,2,wspace=.34,width_ratios=[1.1,1.0])
ax1=fig.add_subplot(gs[0,0]); ax2=fig.add_subplot(gs[0,1])

def panel(ax,label): ax.text(-.13,1.08,label,transform=ax.transAxes,fontweight="bold",fontsize=12,va="bottom")

# panel a
m=(t>=0)&(t<=.008); x=1e3*t[m]
for mode,col in [("slow",SLOW),("fast",FAST)]:
    mean=1e6*S[mode]["mean"][m]; sd=1e6*S[mode]["sd"][m]
    ax1.plot(x,mean,color=col,label=f"{mode} ensemble")
    ax1.fill_between(x,np.maximum(0,mean-sd),mean+sd,color=col,alpha=.16,lw=0)
ax1.axvspan(ci95[0],ci95[1],color="0.7",alpha=.22,label="95% bootstrap interval")
ax1.axvline(1e3*tx,color="0.3",ls="--",lw=1.0,label=r"$t_\times$")
ax1.set_xlim(0,8);ax1.set_ylim(bottom=0)
ax1.set_xlabel("time after trigger (ms)");ax1.set_ylabel(r"ensemble mean energy ($\mu$J)")
ax1.legend(frameon=False,fontsize=7,loc="upper right");panel(ax1,"(a)")

# inset around crossing
ins=ax1.inset_axes([.50,.34,.45,.30])
mi=(t>=.00115)&(t<=.00195); xi=1e3*t[mi]
for mode,col in [("slow",SLOW),("fast",FAST)]:
    mean=1e6*S[mode]["mean"][mi];sd=1e6*S[mode]["sd"][mi]
    ins.plot(xi,mean,color=col,lw=1.0)
    ins.fill_between(xi,np.maximum(0,mean-sd),mean+sd,color=col,alpha=.14,lw=0)
ins.axvspan(ci95[0],ci95[1],color="0.7",alpha=.18)
ins.axvline(1e3*tx,color="0.3",ls="--",lw=.8)
ins.set_title("crossing region",fontsize=7,pad=1);ins.tick_params(labelsize=6)

# panel b
for mode,col,marker in [("slow",SLOW,"o"),("fast",FAST,"s")]:
    xb,yb=binned[mode]
    ax2.plot(xb,yb,color=col,marker=marker,ms=3,label=f"{mode} ensemble")
    ax2.axhline(late_avg[mode],color=col,ls=":",lw=1.0)
ax2.axvspan(40,100,color="0.7",alpha=.12)
ax2.set_xlim(0,100);ax2.set_ylim(0,1.03)
ax2.set_xlabel("time after trigger (ms)");ax2.set_ylabel(r"covariance fraction $E_\Sigma/\overline{E}$")
ax2.text(.04,.92,r"$\overline{E}=E_\mu+E_\Sigma$",transform=ax2.transAxes,fontsize=8)
ax2.text(.96,.10,"40–100 ms\nlate-time window",transform=ax2.transAxes,ha="right",va="bottom",fontsize=7)
ax2.text(98,late_avg["slow"]+.025,f"{late_avg['slow']:.3f}",color=SLOW,ha="right",fontsize=7)
ax2.text(98,late_avg["fast"]-.045,f"{late_avg['fast']:.3f}",color=FAST,ha="right",fontsize=7)
ax2.legend(frameon=False,fontsize=7,loc="center right");panel(ax2,"(b)")

fig.savefig(OUT_FIG/"Figure3_engineered_noise_experiment_reproduced.pdf")
fig.savefig(OUT_FIG/"Figure3_engineered_noise_experiment_reproduced.png",dpi=300)
plt.show()


### What the two panels establish

- **Panel (a):** the energy-order inversion survives continuous externally injected noise at the ensemble level. The bootstrap interval quantifies uncertainty in the crossing time by resampling entire trajectories.
- **Panel (b):** as the first-moment signal decays, an increasing fraction of the ensemble mean energy is stored in the covariance. By the 40–100 ms window, roughly 90% of the reported mean energy is covariance-controlled for both preparation classes.

This is a stochastic persistence result for the engineered drive. It is not evidence that the laboratory noise is an equilibrium thermal bath.


## 6. Numerical regression against the manuscript


In [ ]:
cert=json.loads((CERT/"provenance_summary.json").read_text())
rows=[
    ["crossing_ms",1e3*tx,cert["noisy"]["crossing_ms"],2e-4],
    ["bootstrap95_low_ms",ci95[0],cert["noisy"]["bootstrap_95_ms"][0],2e-4],
    ["bootstrap95_high_ms",ci95[1],cert["noisy"]["bootstrap_95_ms"][1],2e-4],
    ["fast_late_cov_frac",late_avg["fast"],cert["noisy"]["late_fast_covariance_fraction"],2e-6],
    ["slow_late_cov_frac",late_avg["slow"],cert["noisy"]["late_slow_covariance_fraction"],2e-6],
]
reg=pd.DataFrame(rows,columns=["quantity","reproduced","certified","tolerance"])
reg["abs_difference"]=(reg.reproduced-reg.certified).abs();reg["PASS"]=reg.abs_difference<=reg.tolerance
display(reg);assert reg.PASS.all();print("N05 numerical regression: PASS")


## Reader extension

To analyze a new dataset with the same acquisition format, place ten `rapido*.csv` and ten `lento*.csv` files in the configured noisy-data directory. The reconstruction, statistics, bootstrap, and plotting logic require no physics-code edits.
